# Stress Prediction — Ensemble v4

목표는 기존의 강한 ExtraTrees 구조를 유지하면서 서로 다른 예측 성향을 가진 모델을 OOF 기반으로 앙상블하는 것입니다.

구성:
- 원본 ExtraTrees + feature copy + quantile
- Pair nearest-neighbor predictor
- CatBoostRegressor (설치된 경우)
- HistGradientBoostingRegressor
- 모델별 5-Fold OOF MAE 및 예측 상관계수 확인
- 비음수 가중치 합이 1인 coarse-to-fine weight search
- 반올림/override 적용 여부를 별도로 탐색
- 서로 다른 성향의 제출 후보 3개 생성

> CatBoost가 없다면 `pip install catboost` 후 커널을 재시작하세요. 설치하지 않아도 ExtraTrees + HistGradientBoosting + Pair 앙상블은 실행됩니다.


In [1]:
from __future__ import annotations

import os
from itertools import combinations
from pathlib import Path
import warnings

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

try:
    from catboost import CatBoostRegressor
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_SPLITS = 5
N_ESTIMATORS = 1200

# 원본 구조 유지
TREE_QUANTILE = 0.52
PAIR_QUANTILE = 0.48
BASE_PAIR_WEIGHT = 0.20
BASE_OVERRIDE_THRESHOLD = 0.18

WINSORIZE_COLUMNS = [
    "mean_working", "diastolic_blood_pressure", "cholesterol", "glucose"
]

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]

print("CatBoost 사용 가능:", CATBOOST_AVAILABLE)
print("DATA_DIR:", DATA_DIR)


CatBoost 사용 가능: True
DATA_DIR: /Users/bitsaem/Desktop/stress_project/open


## 전처리 및 모델 함수

In [2]:
def compute_iqr_bounds(train_frame: pd.DataFrame, columns: list[str]) -> dict:
    bounds = {}
    for col in columns:
        q1, q3 = train_frame[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        bounds[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return bounds


def apply_winsorize(frame: pd.DataFrame, bounds: dict) -> pd.DataFrame:
    result = frame.copy()
    for col, (lower, upper) in bounds.items():
        result[col] = result[col].clip(lower, upper)
    return result


def add_features(frame: pd.DataFrame, extended: bool = False) -> pd.DataFrame:
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]

    result["bmi"] = result["weight"] / (height_m.pow(2) + 1e-6)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = frame.isna().sum(axis=1)

    # 부스팅 모델에만 사용할 보수적인 상호작용 변수
    if extended:
        result["bmi_squared"] = result["bmi"] ** 2
        result["bp_sum"] = sbp + dbp
        result["bp_product"] = sbp * dbp
        result["cholesterol_glucose_product"] = result["cholesterol"] * result["glucose"]
        result["working_bmi"] = result["mean_working"] * result["bmi"]
        result["working_glucose"] = result["mean_working"] * result["glucose"]
        result["working_cholesterol"] = result["mean_working"] * result["cholesterol"]

        for col in ["mean_working", "cholesterol", "glucose", "weight", "bmi"]:
            result[f"{col}_log1p"] = np.log1p(result[col].clip(lower=0))

    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_dense_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()

    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def extra_trees_prediction_matrix(
    train_features: pd.DataFrame,
    val_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    """원본 ExtraTrees 구조를 유지하고 트리별 예측 행렬을 반환."""
    train_no_gender = train_features.drop(columns=["gender"], errors="ignore")
    val_no_gender = val_features.drop(columns=["gender"], errors="ignore")

    weighted_train = add_feature_copies(train_no_gender)
    weighted_val = add_feature_copies(val_no_gender)

    preprocessor = make_dense_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)

    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,
        min_samples_leaf=1,
        max_features=1,
        criterion="squared_error",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    model.fit(train_matrix, target)

    return np.column_stack([
        tree.predict(val_matrix) for tree in model.estimators_
    ])


def empirical_rank_transform(train_values, test_values):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    test_rank = np.searchsorted(sorted_train, test_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_prediction_matrix(
    train_features: pd.DataFrame,
    val_features: pd.DataFrame,
    target: pd.Series,
    k: int = 1,
) -> np.ndarray:
    """각 feature pair의 최근접 타깃 예측을 열로 갖는 행렬 반환."""
    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )

    target_values = target.to_numpy(dtype=float)
    pair_targets = []

    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )

        if k == 1:
            nearest_idx = np.argmin(distance, axis=1)
            pair_target = target_values[nearest_idx]
        else:
            nearest_idx = np.argpartition(distance, kth=k - 1, axis=1)[:, :k]
            pair_target = np.median(target_values[nearest_idx], axis=1)

        pair_targets.append(pair_target)

    return np.column_stack(pair_targets)


def nearest_neighbor_lookup(train_raw, val_raw, target, numeric_cols):
    train_numeric = train_raw[numeric_cols].fillna(train_raw[numeric_cols].median())
    val_numeric = val_raw[numeric_cols].fillna(train_raw[numeric_cols].median())

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_numeric)
    val_scaled = scaler.transform(val_numeric)

    tree = cKDTree(train_scaled)
    nearest_dist, nearest_idx = tree.query(val_scaled, k=1)
    nearest_target = target.to_numpy(dtype=float)[nearest_idx]
    return nearest_dist, nearest_target


def prepare_catboost_frame(frame: pd.DataFrame):
    result = frame.copy()
    categorical = result.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    for col in categorical:
        result[col] = result[col].fillna("missing").astype(str)
    numeric = [c for c in result.columns if c not in categorical]
    for col in numeric:
        result[col] = result[col].fillna(result[col].median())
    cat_indices = [result.columns.get_loc(c) for c in categorical]
    return result, cat_indices


def catboost_prediction(train_features, val_features, target):
    if not CATBOOST_AVAILABLE:
        return None

    train_cb, cat_indices = prepare_catboost_frame(train_features)
    val_cb, _ = prepare_catboost_frame(val_features)

    model = CatBoostRegressor(
        loss_function="MAE",
        eval_metric="MAE",
        iterations=900,
        learning_rate=0.035,
        depth=7,
        l2_leaf_reg=6.0,
        random_seed=RANDOM_SEED,
        random_strength=0.5,
        bagging_temperature=0.5,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )
    model.fit(train_cb, target, cat_features=cat_indices, verbose=False)
    return model.predict(val_cb)


def hist_gradient_prediction(train_features, val_features, target):
    preprocessor = make_dense_preprocessor(train_features)
    train_matrix = preprocessor.fit_transform(train_features)
    val_matrix = preprocessor.transform(val_features)

    model = HistGradientBoostingRegressor(
        loss="absolute_error",
        learning_rate=0.045,
        max_iter=450,
        max_leaf_nodes=31,
        min_samples_leaf=18,
        l2_regularization=1.0,
        random_state=RANDOM_SEED,
    )
    model.fit(train_matrix, target)
    return model.predict(val_matrix)


## 5-Fold OOF 예측 생성

In [3]:
train_raw = pd.read_csv(DATA_DIR / "train.csv")
y = train_raw[TARGET].reset_index(drop=True)
raw_input = train_raw.drop(columns=[TARGET, ID_COLUMN]).reset_index(drop=True)
numeric_cols = raw_input.select_dtypes(include=["number"]).columns.tolist()

cv = KFold(N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

oof = {
    "extra_trees": np.zeros(len(y)),
    "pair_k1": np.zeros(len(y)),
    "pair_k3": np.zeros(len(y)),
    "hist_gb": np.zeros(len(y)),
}
if CATBOOST_AVAILABLE:
    oof["catboost"] = np.zeros(len(y))

nearest_dist_oof = np.zeros(len(y))
nearest_target_oof = np.zeros(len(y))
fold_cache = []

for fold, (train_idx, val_idx) in enumerate(cv.split(raw_input), start=1):
    train_fold_raw = raw_input.iloc[train_idx].reset_index(drop=True)
    val_fold_raw = raw_input.iloc[val_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)

    bounds = compute_iqr_bounds(train_fold_raw, WINSORIZE_COLUMNS)
    train_w = apply_winsorize(train_fold_raw, bounds)
    val_w = apply_winsorize(val_fold_raw, bounds)

    # 원본 ET/Pair용 feature
    train_base = add_features(train_w, extended=False)
    val_base = add_features(val_w, extended=False)

    # 부스팅용 확장 feature
    train_ext = add_features(train_w, extended=True)
    val_ext = add_features(val_w, extended=True)

    et_matrix = extra_trees_prediction_matrix(train_base, val_base, y_train)
    pair1_matrix = pair_prediction_matrix(train_base, val_base, y_train, k=1)
    pair3_matrix = pair_prediction_matrix(train_base, val_base, y_train, k=3)

    oof["extra_trees"][val_idx] = np.quantile(et_matrix, TREE_QUANTILE, axis=1)
    oof["pair_k1"][val_idx] = np.quantile(pair1_matrix, PAIR_QUANTILE, axis=1)
    oof["pair_k3"][val_idx] = np.quantile(pair3_matrix, PAIR_QUANTILE, axis=1)
    oof["hist_gb"][val_idx] = hist_gradient_prediction(train_ext, val_ext, y_train)

    if CATBOOST_AVAILABLE:
        oof["catboost"][val_idx] = catboost_prediction(train_ext, val_ext, y_train)

    nearest_dist, nearest_target = nearest_neighbor_lookup(
        train_fold_raw, val_fold_raw, y_train, numeric_cols
    )
    nearest_dist_oof[val_idx] = nearest_dist
    nearest_target_oof[val_idx] = nearest_target

    fold_cache.append({
        "train_idx": train_idx,
        "val_idx": val_idx,
    })
    print(f"Fold {fold}/{N_SPLITS} 완료")

print("\n모델별 OOF MAE")
model_score_rows = []
for name, pred in oof.items():
    raw_mae = mean_absolute_error(y, np.clip(pred, 0, 1))
    rounded_mae = mean_absolute_error(y, np.clip(np.round(pred, 2), 0, 1))
    model_score_rows.append({
        "model": name,
        "raw_mae": raw_mae,
        "rounded_mae": rounded_mae,
    })

display(pd.DataFrame(model_score_rows).sort_values("raw_mae"))


Fold 1/5 완료
Fold 2/5 완료
Fold 3/5 완료
Fold 4/5 완료
Fold 5/5 완료

모델별 OOF MAE


,model,raw_mae,rounded_mae
0,extra_trees,0.147479,0.147493
1,pair_k1,0.154009,0.153980
2,pair_k3,0.208569,0.208527
3,hist_gb,0.210192,0.210137
4,catboost,0.215867,0.215873


## 예측 상관계수 확인

In [4]:
oof_frame = pd.DataFrame(oof)
display(oof_frame.corr().round(4))

print("상관계수가 지나치게 높은 모델끼리만 섞으면 앙상블 효과가 작습니다.")


,extra_trees,pair_k1,pair_k3,hist_gb,catboost
extra_trees,1.0000,0.8915,0.6533,0.6302,0.6211
pair_k1,0.8915,1.0000,0.7465,0.5856,0.5697
pair_k3,0.6533,0.7465,1.0000,0.5470,0.5109
hist_gb,0.6302,0.5856,0.5470,1.0000,0.8525
catboost,0.6211,0.5697,0.5109,0.8525,1.0000


상관계수가 지나치게 높은 모델끼리만 섞으면 앙상블 효과가 작습니다.


## OOF 기반 앙상블 가중치 탐색

In [5]:
# Pair k=1과 k=3 중 OOF가 더 좋은 하나만 weight search에 사용
pair_name = min(
    ["pair_k1", "pair_k3"],
    key=lambda name: mean_absolute_error(y, np.clip(oof[name], 0, 1)),
)

search_model_names = ["extra_trees", "hist_gb", pair_name]
if CATBOOST_AVAILABLE:
    search_model_names.insert(1, "catboost")

print("가중치 탐색 모델:", search_model_names)


def generate_weight_vectors(n_models: int, step: float):
    units = int(round(1.0 / step))
    for parts in combinations(range(units + n_models - 1), n_models - 1):
        bars = (-1,) + parts + (units + n_models - 1,)
        counts = [bars[i + 1] - bars[i] - 1 for i in range(n_models)]
        yield np.array(counts, dtype=float) / units


def apply_postprocess(pred, round_decimals, threshold):
    result = np.clip(pred.copy(), 0, 1)
    if round_decimals is not None:
        result = np.round(result, round_decimals)
    if threshold is not None:
        mask = nearest_dist_oof < threshold
        replacement = nearest_target_oof[mask]
        if round_decimals is not None:
            replacement = np.round(replacement, round_decimals)
        result[mask] = np.clip(replacement, 0, 1)
    return result

pred_matrix = np.column_stack([oof[name] for name in search_model_names])
search_rows = []

# coarse search: step 0.05
for weights in generate_weight_vectors(len(search_model_names), step=0.05):
    blended = pred_matrix @ weights

    for decimals in [None, 2]:
        for threshold in [None, 0.12, 0.15, 0.18, 0.20, 0.22]:
            final_pred = apply_postprocess(blended, decimals, threshold)
            score = mean_absolute_error(y, final_pred)

            row = {
                "mae": score,
                "round_decimals": decimals,
                "override_threshold": threshold,
            }
            row.update({name: weight for name, weight in zip(search_model_names, weights)})
            search_rows.append(row)

search_df = pd.DataFrame(search_rows).sort_values("mae").reset_index(drop=True)
display(search_df.head(25))

best_row = search_df.iloc[0]
print("최적 OOF MAE:", best_row["mae"])


가중치 탐색 모델: ['extra_trees', 'catboost', 'hist_gb', 'pair_k1']


,mae,round_decimals,override_threshold,extra_trees,catboost,hist_gb,pair_k1
0,0.147087,2.0,0.15,0.85,0.0,0.0,0.15
1,0.147087,2.0,0.22,0.85,0.0,0.0,0.15
2,0.147087,2.0,0.20,0.85,0.0,0.0,0.15
3,0.147087,2.0,0.18,0.85,0.0,0.0,0.15
4,0.147087,2.0,0.12,0.85,0.0,0.0,0.15
5,0.147093,2.0,0.18,0.80,0.0,0.0,0.20
6,0.147093,2.0,0.20,0.80,0.0,0.0,0.20
7,0.147093,2.0,0.22,0.80,0.0,0.0,0.20
8,0.147093,2.0,0.12,0.80,0.0,0.0,0.20
9,0.147093,2.0,0.15,0.80,0.0,0.0,0.20


최적 OOF MAE: 0.14708666666666667


## 제출 후보 선택

CV 1등만 제출하면 Public에서 과적합될 수 있으므로 다음 세 종류를 생성합니다.

1. **CV best**: OOF MAE 최저 조합
2. **ET-heavy**: 기존 리더보드 강점이 있는 ExtraTrees 비중을 높인 보수적 조합
3. **Diverse ensemble**: CatBoost/HGB 비중을 확보한 다양성 조합


In [6]:
def row_to_config(row):
    return {
        "weights": {name: float(row[name]) for name in search_model_names},
        "round_decimals": None if pd.isna(row["round_decimals"]) else int(row["round_decimals"]),
        "override_threshold": None if pd.isna(row["override_threshold"]) else float(row["override_threshold"]),
        "oof_mae": float(row["mae"]),
    }

configs = []

# 1) CV 최적
configs.append(("cv_best", row_to_config(search_df.iloc[0])))

# 2) ET 비중 0.45 이상인 최상위
et_heavy_df = search_df[search_df["extra_trees"] >= 0.45]
if len(et_heavy_df):
    configs.append(("et_heavy", row_to_config(et_heavy_df.iloc[0])))

# 3) 부스팅 비중이 일정 수준 이상인 최상위
boost_cols = [c for c in ["catboost", "hist_gb"] if c in search_model_names]
if boost_cols:
    boost_weight = search_df[boost_cols].sum(axis=1)
    diverse_df = search_df[boost_weight >= 0.35]
    if len(diverse_df):
        configs.append(("diverse", row_to_config(diverse_df.iloc[0])))

# 중복 제거 후 최대 3개
unique_configs = []
seen = set()
for name, config in configs:
    key = (
        tuple(sorted(config["weights"].items())),
        config["round_decimals"],
        config["override_threshold"],
    )
    if key not in seen:
        seen.add(key)
        unique_configs.append((name, config))

configs = unique_configs[:3]
for name, config in configs:
    print("\n", name, config)



 cv_best {'weights': {'extra_trees': 0.85, 'catboost': 0.0, 'hist_gb': 0.0, 'pair_k1': 0.15}, 'round_decimals': 2, 'override_threshold': 0.15, 'oof_mae': 0.14708666666666667}

 diverse {'weights': {'extra_trees': 0.55, 'catboost': 0.0, 'hist_gb': 0.35, 'pair_k1': 0.1}, 'round_decimals': 2, 'override_threshold': 0.22, 'oof_mae': 0.15844333333333335}


## 전체 데이터 재학습 및 제출 파일 생성

In [7]:
test_raw = pd.read_csv(DATA_DIR / "test.csv")
submission_template = pd.read_csv(DATA_DIR / "sample_submission.csv")
test_input = test_raw.drop(columns=[ID_COLUMN]).reset_index(drop=True)

bounds = compute_iqr_bounds(raw_input, WINSORIZE_COLUMNS)
train_w = apply_winsorize(raw_input, bounds)
test_w = apply_winsorize(test_input, bounds)

train_base = add_features(train_w, extended=False)
test_base = add_features(test_w, extended=False)
train_ext = add_features(train_w, extended=True)
test_ext = add_features(test_w, extended=True)

print("ExtraTrees 전체 학습")
et_matrix_test = extra_trees_prediction_matrix(train_base, test_base, y)
test_predictions = {
    "extra_trees": np.quantile(et_matrix_test, TREE_QUANTILE, axis=1),
}

print("Pair 전체 학습")
selected_pair_k = 1 if pair_name == "pair_k1" else 3
pair_matrix_test = pair_prediction_matrix(train_base, test_base, y, k=selected_pair_k)
test_predictions[pair_name] = np.quantile(pair_matrix_test, PAIR_QUANTILE, axis=1)

print("HistGradientBoosting 전체 학습")
test_predictions["hist_gb"] = hist_gradient_prediction(train_ext, test_ext, y)

if CATBOOST_AVAILABLE:
    print("CatBoost 전체 학습")
    test_predictions["catboost"] = catboost_prediction(train_ext, test_ext, y)

nearest_dist_test, nearest_target_test = nearest_neighbor_lookup(
    raw_input, test_input, y, numeric_cols
)

saved_paths = []
for rank, (name, config) in enumerate(configs, start=1):
    pred = np.zeros(len(test_input), dtype=float)
    for model_name, weight in config["weights"].items():
        pred += weight * test_predictions[model_name]

    pred = np.clip(pred, 0, 1)
    decimals = config["round_decimals"]
    threshold = config["override_threshold"]

    if decimals is not None:
        pred = np.round(pred, decimals)

    if threshold is not None:
        mask = nearest_dist_test < threshold
        replacement = nearest_target_test[mask]
        if decimals is not None:
            replacement = np.round(replacement, decimals)
        pred[mask] = np.clip(replacement, 0, 1)

    output = submission_template.copy()
    output[TARGET] = pred

    path = OUTPUT_DIR / f"submit_ensemble_v4_rank{rank}_{name}.csv"
    output.to_csv(path, index=False, encoding="utf-8")
    saved_paths.append(path)

    print("\nSaved:", path)
    print("Config:", config)
    print("Prediction range:", float(pred.min()), float(pred.max()))
    display(output.head())

print("\n생성 완료")
for path in saved_paths:
    print(path)


ExtraTrees 전체 학습
Pair 전체 학습
HistGradientBoosting 전체 학습
CatBoost 전체 학습

Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_ensemble_v4_rank1_cv_best.csv
Config: {'weights': {'extra_trees': 0.85, 'catboost': 0.0, 'hist_gb': 0.0, 'pair_k1': 0.15}, 'round_decimals': 2, 'override_threshold': 0.15, 'oof_mae': 0.14708666666666667}
Prediction range: 0.0 1.0


,ID,stress_score
0,TEST_0000,0.60
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.51
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_ensemble_v4_rank2_diverse.csv
Config: {'weights': {'extra_trees': 0.55, 'catboost': 0.0, 'hist_gb': 0.35, 'pair_k1': 0.1}, 'round_decimals': 2, 'override_threshold': 0.22, 'oof_mae': 0.15844333333333335}
Prediction range: 0.0 1.0


,ID,stress_score
0,TEST_0000,0.51
1,TEST_0001,0.97
2,TEST_0002,0.22
3,TEST_0003,0.50
4,TEST_0004,0.54



생성 완료
/Users/bitsaem/Desktop/stress_project/8_6/submit_ensemble_v4_rank1_cv_best.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_ensemble_v4_rank2_diverse.csv
